# Project Milestone 2

In [7]:
# Importing Pandas packages required for this excercise.

import pandas as pd
import seaborn as sns
import matplotlib

import matplotlib.dates as mdates
from matplotlib.dates import DateFormatter
import matplotlib.ticker as tick


In [28]:
# loading covid CSV file for data transformation and cleaning

covid = pd.read_csv("./time_series_covid_19_confirmed.csv")
covid.head()

,Province/State,Country/Region,Lat,Long,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,...,4/4/20,4/5/20,4/6/20,4/7/20,4/8/20,4/9/20,4/10/20,4/11/20,4/12/20,4/13/20
0,NaN,Afghanistan,33.0000,65.0000,0,0,0,0,0,0,...,299,349,367,423,444,484,521,555,607,665
1,NaN,Albania,41.1533,20.1683,0,0,0,0,0,0,...,333,361,377,383,400,409,416,433,446,467
2,NaN,Algeria,28.0339,1.6596,0,0,0,0,0,0,...,1251,1320,1423,1468,1572,1666,1761,1825,1914,1983
3,NaN,Andorra,42.5063,1.5218,0,0,0,0,0,0,...,466,501,525,545,564,583,601,601,638,646
4,NaN,Angola,-11.2027,17.8739,0,0,0,0,0,0,...,10,14,16,17,19,19,19,19,19,19


In [30]:
# Dropping lat.lon columns not required in this project from file

coviddrop = covid.drop(['Lat','Long'],axis = 1)
coviddrop

,Province/State,Country/Region,1/22/20,1/23/20,1/24/20,1/25/20,1/26/20,1/27/20,1/28/20,1/29/20,...,4/4/20,4/5/20,4/6/20,4/7/20,4/8/20,4/9/20,4/10/20,4/11/20,4/12/20,4/13/20
0,NaN,Afghanistan,0,0,0,0,0,0,0,0,...,299,349,367,423,444,484,521,555,607,665
1,NaN,Albania,0,0,0,0,0,0,0,0,...,333,361,377,383,400,409,416,433,446,467
2,NaN,Algeria,0,0,0,0,0,0,0,0,...,1251,1320,1423,1468,1572,1666,1761,1825,1914,1983
3,NaN,Andorra,0,0,0,0,0,0,0,0,...,466,501,525,545,564,583,601,601,638,646
4,NaN,Angola,0,0,0,0,0,0,0,0,...,10,14,16,17,19,19,19,19,19,19
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
259,Saint Pierre and Miquelon,France,0,0,0,0,0,0,0,0,...,0,1,1,1,1,1,1,1,1,1
260,NaN,South Sudan,0,0,0,0,0,0,0,0,...,0,1,1,2,2,3,4,4,4,4
261,NaN,Western Sahara,0,0,0,0,0,0,0,0,...,0,4,4,4,4,4,4,4,6,6
262,NaN,Sao Tome and Principe,0,0,0,0,0,0,0,0,...,0,0,4,4,4,4,4,4,4,4


In [32]:
# Selecting 4 countries that has highest number of cases

covidMelt = coviddrop.melt(id_vars = ['Country/Region'])
coviddropNan = covidMelt.dropna()
coviddropvariable = coviddropNan[coviddropNan['variable'] != 'Province/State']
covidtopCountries = coviddropvariable.groupby(['Country/Region'])['value'].sum().reset_index().sort_values(by = 'value',ascending = False)
covidtopCountries[0:5] 

,Country/Region,value
171,US,6278243
36,China,5262621
84,Italy,2977220
156,Spain,2574182
65,Germany,1986317


In [17]:
Covidtop4countries = coviddropvariable.loc[coviddropvariable['Country/Region'].isin(['US','Brazil','Russia','Spain'])]
set(Covidtop4countries['Country/Region'])

{'Brazil', 'Russia', 'Spain', 'US'}

In [34]:
# Renaming country date columns and Parsing date and grouping data

import numpy as np
pd.set_option('mode.chained_assignment',None)
Covidtop4countries.columns = ['Country','Date','No of Cases']
Covidtop4countries['Date'] = pd.to_datetime(Covidtop4countries.Date)
Covidtop4countriesGroup = Covidtop4countries.groupby(['Country',pd.Grouper(key = 'Date', freq = 'SM')])['No of Cases'].sum().reset_index().sort_values('Date')
Covidtop4countriesGroup
Covidtop4countriesGroup1 = Covidtop4countriesGroup[Covidtop4countriesGroup['Date'] < '2020-06-30']
Covidtop4countriesGroup1
Covidtop4countriesGroup1['Parsed Date'] = Covidtop4countriesGroup1['Date'].astype(str)
Covidtop4countriesGroup1['Parsed Date'] =  Covidtop4countriesGroup1['Parsed Date'].str[6:]
Covidtop4countriesGroup1['Parsed Date'] = Covidtop4countriesGroup1['Parsed Date'].replace({'-':'/'}, regex=True)
Covidtop4countriesGroup1[0:10]

/var/folders/18/ygzmjvj90m72g8_1d1sfk1b80000gn/T/ipykernel_43399/2231736262.py:7: FutureWarning: 'SM' is deprecated and will be removed in a future version, please use 'SME' instead.
  Covidtop4countriesGroup = Covidtop4countries.groupby(['Country',pd.Grouper(key = 'Date', freq = 'SM')])['No of Cases'].sum().reset_index().sort_values('Date')


,Country,Date,No of Cases,Parsed Date
0,Brazil,2020-01-15,0,1/15
12,Spain,2020-01-15,0,1/15
18,US,2020-01-15,31,1/15
6,Russia,2020-01-15,0,1/15
13,Spain,2020-01-31,20,1/31
7,Russia,2020-01-31,30,1/31
19,US,2020-01-31,161,1/31
1,Brazil,2020-01-31,0,1/31
8,Russia,2020-02-15,28,2/15
2,Brazil,2020-02-15,3,2/15


In [36]:
# Applying stack function and transforming the columns accordingly.

covidUS = covid[covid['Country/Region'] == 'US']
covidLatLong = covidUS.drop(['Lat','Long'],axis = 1)
Covidtranform = covidLatLong.stack()
CovidtranformF = Covidtranform.reset_index()
CovidtranformF

,level_0,level_1,0
0,225,Country/Region,US
1,225,1/22/20,1
2,225,1/23/20,1
3,225,1/24/20,2
4,225,1/25/20,2
...,...,...,...
79,225,4/9/20,461437
80,225,4/10/20,496535
81,225,4/11/20,526396
82,225,4/12/20,555313


In [23]:
# renaming columns and sorting the data afterwards
CovidtranformF = CovidtranformF[CovidtranformF['level_1'] != 'Country/Region']
CovidtranformFilter = CovidtranformF[['level_1',0]]
CovidtranformFilter.columns = ['Date','No of Cases']
CovidtranformFilter = CovidtranformFilter.sort_values(by = 'No of Cases',ascending = False)
CovidtranformFilter

,Date,No of Cases
83,4/13/20,580619
82,4/12/20,555313
81,4/11/20,526396
80,4/10/20,496535
79,4/9/20,461437
...,...,...
6,1/27/20,5
3,1/24/20,2
4,1/25/20,2
2,1/23/20,1


In [38]:
Covidsort = CovidtranformFilter.sort_values(by = 'No of Cases',ascending = False)
Covidsort


,Date,No of Cases
83,4/13/20,580619
82,4/12/20,555313
81,4/11/20,526396
80,4/10/20,496535
79,4/9/20,461437
...,...,...
6,1/27/20,5
3,1/24/20,2
4,1/25/20,2
2,1/23/20,1
